#### 감정 분석 자연어 처리
1. data 폴더 안에 ratings_train.txt 로드
2. 상위 500개 데이터만 추출
3. 리뷰 데이터와 감정 데이터로 나눠준다.
4. 리뷰 데이터를 토큰화(komoran함수 이용)
5. Word2Vec 학습
    - window -> 5
    - epochs -> 100
    - min_count -> 2
    - sg -> 1
    - seed -> 42
6. 벡터화(Word2Vec, 단위 벡터의 평균)
7. 분류 모델 (SVC, Logistic)
8. Train, Test를 이용하여 2개의 모델 중 성능이 높은 모델이 무엇인가?
9. 단위 벡터의 평균의 성능과 단위벡터 + 중요도 평균의 성능의 차이를 확인

In [50]:
import pandas as pd
from gensim.models import Word2Vec
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [51]:
df = pd.read_csv('../data_git/data_NLP/ratings_train.txt', sep = '\t')

In [52]:
# 토큰화 함수 생성 
# komoran 사용 (konlpy 설치가 되어있는 경우)
# 설치가 되어있지 않은 경우에는 split()을 이용하여 토큰화 
def build_tokenize():
    try:
        # 라이브러리 로드 -> 라이브러리가 존재하면 코드들 실행 
        from konlpy.tag import Komoran
        komoran = Komoran()
        allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'SL', 'MAG']
        def tokenize(text):
            tokens = []
            for word, pos in komoran.pos(text):
                if pos in allow_pos:
                    tokens.append(word)
            return tokens
        # tokenize 함수를 결과로 되돌려준다. 
        return tokenize
    except Exception as e:
        print("Komoran 사용 불가 : ", e)
        return lambda x : x.split()
    
tokenize = build_tokenize()

In [68]:
reviews = df['document'][ : 500].values

In [69]:
reviews

array(['아 더빙.. 진짜 짜증나네요 목소리', '흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나',
       '너무재밓었다그래서보는것을추천한다', '교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정',
       '사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다',
       '막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...별반개도 아까움.',
       '원작의 긴장감을 제대로 살려내지못했다.',
       '별 반개도 아깝다 욕나온다 이응경 길용우 연기생활이몇년인지..정말 발로해도 그것보단 낫겟다 납치.감금만반복반복..이드라마는 가족도없다 연기못하는사람만모엿네',
       '액션이 없는데도 재미 있는 몇안되는 영화',
       '왜케 평점이 낮은건데? 꽤 볼만한데.. 헐리우드식 화려함에만 너무 길들여져 있나?',
       '걍인피니트가짱이다.진짜짱이다♥', '볼때마다 눈물나서 죽겠다90년대의 향수자극!!허진호는 감성절제멜로의 달인이다~',
       '울면서 손들고 횡단보도 건널때 뛰쳐나올뻔 이범수 연기 드럽게못해',
       '담백하고 깔끔해서 좋다. 신문기사로만 보다 보면 자꾸 잊어버린다. 그들도 사람이었다는 것을.',
       '취향은 존중한다지만 진짜 내생에 극장에서 본 영화중 가장 노잼 노감동임 스토리도 어거지고 감동도 어거지',
       'ㄱ냥 매번 긴장되고 재밋음ㅠㅠ',
       '참 사람들 웃긴게 바스코가 이기면 락스코라고 까고바비가 이기면 아이돌이라고 깐다.그냥 까고싶어서 안달난것처럼 보인다',
       '굿바이 레닌 표절인것은 이해하는데 왜 뒤로 갈수록 재미없어지냐',
       '이건 정말 깨알 캐스팅과 질퍽하지않은 산뜻한 내용구성이 잘 버무러진 깨알일드!!♥',
       '약탈자를 위한 변명, 이라. 저놈들은 착한놈들 절대 아닌걸요.',
       '나름 심오한 뜻도 있는 듯. 그냥 학

In [63]:
target = reviews = df['label'][ : 500].values

In [70]:
X_tokens = [tokenize(review) for review in reviews]

In [71]:
# Word2Vec을 이용하여 학습(Skip-gram 방식)
w2v = Word2Vec(
    sentences= X_tokens, 
    vector_size= 100, 
    window = 5, 
    min_count= 1, 
    sg = 1, 
    epochs= 100, 
    seed = 42, 
    workers=2
)

# w2v 객체에서 wv 속성을 자주 사용하기에 변수에 따로 저장 
wv = w2v.wv

In [72]:
# 문장을 임베딩하는 함수를 생성 -> 벡터화 
# 단위 벡터의 평균을 구하는 함수
def sent_embed_mean(tokens):
    vecs = []
    for word in tokens:
        if word in wv.index_to_key:
            vecs.append(wv[word])
    result = np.mean(vecs, axis=0) if vecs else np.zeros(wv.vector_size)
    return result


# Word2Vec과 TF-IDF를 융합하여 임배딩 처리 함수 생성
# 문맥상에서 단어의 예측 벡터와 전체 문서에서 특정 단어들의 중요도를 결합한 벡터 데이터
# TF-IDF 벡터화 행렬 생성
tfidf_vec = TfidfVectorizer(
    tokenizer=tokenize,
    lowercase=False
).fit(reviews)

idf = dict(
    zip(
        # get_feature_names_out() -> tfidf에서 사용된 단어들의 목록
        tfidf_vec.get_feature_names_out(),
        # idf_: 중요도
        tfidf_vec.idf_
    )
)

# 단어 별 단위 벡터의 평균과 idf를 곱한다.
def sent_embed_tfidf(tokens):
    vecs = []
    weight = []
    for word in tokens:
        # tokens에 각각의 단어가 Word2Vec과 TF-IDF에 존재한다면
        if word in wv.key_to_index and word in idf:
            # vecs -> 단위벡터와 중요도를 곱한 값을 vecs에 추가
            vecs.append(wv[word] * idf[word])
            # weight -> 중요도 데이터를 추가
            weight.append(idf[word])
    # vecs의 데이터가 존재하지 않는다면 -> tokens 안에 단어는 존재하지만 Word2Vec이나 TF-IDF에 단어가 존재하지 않을 때
    if not vecs:
        # 희소 행렬을 되돌려준다. 0행렬
        result = np.zeros(wv.vector_size)
    else:
        result = np.sum(vecs, axis=0) / (np.sum(weight) + 1e-9)
    return result

/Users/eunseo/Documents/data_boot/venv311/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [75]:
X_embed1 = [sent_embed_mean(token) for token in X_tokens]

In [76]:
X_embed2 = [sent_embed_tfidf(token) for token in X_tokens]

- svc

In [89]:
def run_model_svc(X, Y, test_size = 0.2):
    svc = SVC(random_state=42)
    # X는 독립 변수
    # Y는 종속 변수
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=test_size
    )
    # 모델에 학습
    svc.fit(X_train, Y_train)
    # 학습된 모델에 예측 값
    y_pred = svc.predict(X_test)
    print('정확도 : ', round(
        accuracy_score(y_pred, Y_test), 4
    ))
    print('분류 레포트 : ') 
    print(classification_report(y_pred, Y_test))

In [90]:
# svc
run_model_svc(X_embed1, target)

정확도 :  0.73
분류 레포트 : 
              precision    recall  f1-score   support

           0       0.69      0.70      0.70        44
           1       0.76      0.75      0.76        56

    accuracy                           0.73       100
   macro avg       0.73      0.73      0.73       100
weighted avg       0.73      0.73      0.73       100



In [91]:
run_model_svc(X_embed2, target)

정확도 :  0.68
분류 레포트 : 
              precision    recall  f1-score   support

           0       0.66      0.66      0.66        47
           1       0.70      0.70      0.70        53

    accuracy                           0.68       100
   macro avg       0.68      0.68      0.68       100
weighted avg       0.68      0.68      0.68       100



- logistic regression

In [86]:
# Logistic
def run_model_clf(X, Y, test_size = 0.2):
    clf = LogisticRegression(random_state=42)
    # X는 독립 변수
    # Y는 종속 변수
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=test_size
    )
    # 모델에 학습
    clf.fit(X_train, Y_train)
    # 학습된 모델에 예측 값
    y_pred = clf.predict(X_test)
    print('정확도 : ', round(
        accuracy_score(y_pred, Y_test), 4
    ))
    print('분류 레포트 : ') 
    print(classification_report(y_pred, Y_test))

In [87]:
run_model_clf(X_embed1, target)

정확도 :  0.72
분류 레포트 : 
              precision    recall  f1-score   support

           0       0.64      0.76      0.70        42
           1       0.80      0.69      0.74        58

    accuracy                           0.72       100
   macro avg       0.72      0.73      0.72       100
weighted avg       0.73      0.72      0.72       100



In [88]:
run_model_clf(X_embed2, target)

정확도 :  0.68
분류 레포트 : 
              precision    recall  f1-score   support

           0       0.67      0.65      0.66        48
           1       0.69      0.71      0.70        52

    accuracy                           0.68       100
   macro avg       0.68      0.68      0.68       100
weighted avg       0.68      0.68      0.68       100



In [83]:
def predict_sentence_list(sentences, model, vec_type = 'mean'):
    # sentences : 문장들의 리스트
    # 문장들을 토큰화 -> 임베딩
    X_test = []
    for sent in sentences:
        # token()함수를 호출하여 토큰화
        tokens = tokenize(sent)
        # 토큰화 된 문장을 sent_embed_mean 함수에 입력하여 호출 (단위 벡터의 평균)
        if vec_type == 'mean':
            vec = sent_embed_mean(tokens)
        elif vec == 'tfidf':
            vec = sent_embed_tfidf(tokens)
        X_test.append(vec)
    
    preds = model.predict(X_test)
    result = []
    for sent, pred in zip(sentences, preds):
        label = "긍정" if pred == 1 else "부정"
        result.append([sent, label])
    return result
